# Limit Order Book (LOB) Liquidity Prediction

### Market Trading Algorithmic Signals — Banking-AI

This notebook explains how traders predict the "ease of trading" (liquidity) using the order book:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of market trading and algorithmic signals terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the market trading and algorithmic signals prediction task and why regression modelling supports this banking decision.
- Implement a regression workflow and evaluate predictive accuracy using domain-appropriate error metrics.
- Evaluate whether the prediction error is acceptable for the operational decision it supports.

**Estimated time:** 35–45 minutes

**Collection context:** Volatility forecasting, trading signals, price prediction, and quantitative market analysis.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** When you buy a stock, you don't just see one price. You see a "Bid" (the highest price someone will pay) and an "Ask" (the lowest price someone will sell for). The difference is the **Spread**. In a liquid market, the spread is tiny. In a "thirsty" or illiquid market, the spread is wide, making it expensive to trade. Predicting when liquidity will dry up helps traders avoid high costs.

**Input data used:** Bid-Ask Spread, Bid Volume (at 5 levels), Ask Volume (at 5 levels), and Trade Intensity.

**What we predict:** Future Spread (`Target_Spread`).

**ML method used:** Random Forest Regressor.

**Learning goal:** Learn how to use micro-market structure data (LOB) for forecasting.

**Primary stakeholders:** quantitative analysts, portfolio managers, risk officers, and trading desk supervisors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Limit Order Book (LOB) Data

We simulate 5,000 snapshots of a market's order book.

In [ ]:
n_snapshots = 5000
rng = RNG

# 1. Simulate Volumes at various price levels (Level 1 to Level 5)
bid_vol = rng.poisson(100, (n_snapshots, 5))
ask_vol = rng.poisson(100, (n_snapshots, 5))

# 2. Simulate the Spread (The gap between best bid and best ask)
# Spread tends to widen when volumes are low or lopsided
total_vol = bid_vol.sum(axis=1) + ask_vol.sum(axis=1)
imbalance = np.abs(bid_vol[:, 0] - ask_vol[:, 0]) / (bid_vol[:, 0] + ask_vol[:, 0])

spread = (1 / (total_vol / 1000)) + (imbalance * 0.5) + rng.normal(0.05, 0.01, n_snapshots)
spread = spread.clip(0.01, 1.0)

df = pd.DataFrame({
    'bid_vol_l1': bid_vol[:, 0],
    'ask_vol_l1': ask_vol[:, 0],
    'total_vol': total_vol,
    'imbalance': imbalance,
    'current_spread': spread
})

# Target: The spread in the next snapshot (t+1)
df['target_spread'] = df['current_spread'].shift(-1)
df.dropna(inplace=True)

df.head()

## Step 4 — Exploratory Data Analysis

EDA

Does high imbalance really lead to a wider spread?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='imbalance', y='current_spread', alpha=0.4)
plt.title('Order Imbalance vs. Bid-Ask Spread')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot titled "Order Imbalance vs. Bid-Ask Spread". The chart highlights spatial separation or clustering patterns in the data.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('target_spread', axis=1)
y = df['target_spread']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the training-set mean ---
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train, y_train)
baseline_pred_bl = baseline_reg.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred_bl))
print(f"Baseline RMSE (predict-mean): {baseline_rmse:.4f}")
print("Any useful model must produce a lower RMSE.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")

In [ ]:
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.sort_values().plot(kind='barh', color='#2A9D8F')
plt.title('LOB Features Driving Future Spread')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "LOB Features Driving Future Spread". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- **Current Spread** is the best predictor of future spread, but **Order Imbalance** provides a significant early warning signal.
- When the bid and ask volumes are heavily lopsided, the spread tends to widen shortly after.

**Market Context:** Market makers (the banks providing liquidity) use these models to adjust their quotes. If they predict the spread will widen, they will pull back their orders to protect themselves from "Toxic Flow" (traders who know more than they do), which can cause sudden price drops.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: inspect predictions at different points ---
print("Operational demonstration — model predictions across the test range:\n")
low_idx  = int(np.argmin(y_test.values))
high_idx = int(np.argmax(y_test.values))
mid_idx  = int(np.argsort(y_test.values)[len(y_test) // 2])

for label, idx in [("Low-end", low_idx), ("Mid-range", mid_idx), ("High-end", high_idx)]:
    actual = y_test.values[idx]
    pred   = y_pred[idx]
    print(f"  {label}  actual: {actual:.4f}  predicted: {pred:.4f}  error: {abs(actual-pred):.4f}")

print("\nPractitioners would use these predictions as one input among several.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end market trading and algorithmic signals workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Algorithmic trading models can amplify market volatility and systemic risk if deployed without safeguards.
- Backtested performance often overstates real-world returns due to look-ahead bias and transaction costs.
- Automated trading decisions must include human oversight and circuit breakers.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in market trading and algorithmic signals?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.